In [ ]:
import boto3
from datetime import datetime

# Print execution info
print(f"Execution started at: {datetime.now()}")

# Get current IAM role
sts = boto3.client('sts')
identity = sts.get_caller_identity()
print(f"Current Role ARN: {identity['Arn']}")
print(f"Account ID: {identity['Account']}")

In [ ]:
# Parameters cell - will be replaced by Papermill
model_s3_uri = None
sklearn_version = "1.2-1"
python_version = "py3"
inference_instance_type = "ml.m5.large"
region = None  # AWS region - injected via domain.region
batch_input_file = "input.csv"  # overridden via {env.BATCH_INPUT_FILE}


In [ ]:
import os
print(f"IMAGE_VERSION: {os.environ.get('IMAGE_VERSION', 'Not set')}")
print(f"SAGEMAKER_INTERNAL_IMAGE_URI: {os.environ.get('SAGEMAKER_INTERNAL_IMAGE_URI', 'Not set')}")

In [ ]:
from datetime import datetime
print(f"Execution Start Time (UTC): {datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S UTC')}")


In [ ]:
# Pin sagemaker SDK v2 - the execution environment ships v3 which removed
# the sagemaker.sklearn module used by this notebook.
import subprocess, sys
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '-q', 'sagemaker>=2.200,<3']
)
print('✅ sagemaker SDK v2 installed')

In [ ]:
import boto3
import sagemaker
from sagemaker_studio import Project
from sagemaker.sklearn import SKLearnModel

print("✅ Imports successful")

In [ ]:
from sagemaker_studio import Project

# Get project connections dynamically
proj = Project()

# Get region
if not region:
    region = boto3.Session().region_name
print(f"✅ Region: {region}")

# Get S3 bucket
s3_shared_conn = proj.connection('default.s3_shared')
bucket = s3_shared_conn.data.s3_uri.split('/')[2]  # Extract bucket name from s3://BUCKET/...
s3_prefix = s3_shared_conn.data.s3_uri.replace(f's3://{bucket}/', '')  # S3 prefix from connection
print(f"✅ S3 Bucket: {bucket}")

# Get IAM role (default.iam for IAM domains, project.iam for IdC domains)
try:
    iam_conn = proj.connection('default.iam')
except (AttributeError, Exception):
    iam_conn = proj.connection('project.iam')
role = iam_conn.iam_role
print(f"✅ IAM Role: {role}")

# Create SageMaker session with explicit bucket to avoid CreateBucket permission issues
boto_session = boto3.Session(region_name=region)
# Use s3_prefix (minus trailing slash) as default_bucket_prefix for IdC compatibility
session = sagemaker.Session(boto_session=boto_session, default_bucket=bucket, default_bucket_prefix=s3_prefix.rstrip("/"))
print(f"✅ SageMaker session created with bucket: {bucket}")


In [ ]:
# Cleanup old endpoints from previous runs before creating a new one
print("\n🧹 Cleaning up old endpoints...")

import botocore

sm_client = boto3.client('sagemaker', region_name=region)
name_prefixes = ('sagemaker-scikit-learn', 'realistic-classifier')

endpoints = []
seen = set()
paginator = sm_client.get_paginator('list_endpoints')
for page in paginator.paginate(PaginationConfig={'PageSize': 100}):
    for endpoint in page.get('Endpoints', []):
        endpoint_name = endpoint['EndpointName']
        if endpoint_name in seen:
            continue
        if any(prefix in endpoint_name for prefix in name_prefixes):
            seen.add(endpoint_name)
            endpoints.append(endpoint)

if not endpoints:
    print("   No old endpoints to clean up")
else:
    failed = []
    for endpoint in endpoints:
        endpoint_name = endpoint['EndpointName']
        print(f"   Deleting {endpoint_name} ({endpoint.get('EndpointStatus')})...")
        try:
            details = sm_client.describe_endpoint(EndpointName=endpoint_name)
            config_name = details['EndpointConfigName']
            if details['EndpointStatus'] != 'Deleting':
                sm_client.delete_endpoint(EndpointName=endpoint_name)
            sm_client.get_waiter('endpoint_deleted').wait(
                EndpointName=endpoint_name,
                WaiterConfig={'Delay': 15, 'MaxAttempts': 40},
            )
            try:
                sm_client.delete_endpoint_config(EndpointConfigName=config_name)
            except botocore.exceptions.ClientError as config_error:
                error_code = config_error.response['Error']['Code']
                if error_code not in ('ValidationException', 'ResourceNotFound'):
                    raise
                print(f"   Endpoint config already gone: {config_name}")
            print(f"   Deleted {endpoint_name}")
        except Exception as e:
            print(f"   Could not delete {endpoint_name}: {e}")
            failed.append(endpoint_name)
    if failed:
        raise RuntimeError(f"Failed to delete endpoints: {failed}")
    print(f"   ✅ Cleaned up {len(endpoints)} endpoints")

In [ ]:
# Get model from S3
print("\n📦 Using model from S3...")

if not model_s3_uri:
    raise ValueError("model_s3_uri parameter is required")

model_data = model_s3_uri
print(f"   Model S3 URI: {model_data}")
print("   ✅ Model location retrieved")

In [ ]:
# Deploy the same model to a live endpoint and prepare it for batch transform
print("\n🚀 Deploying live endpoint and batch transform...")

# Download inference.py from S3 to temp directory
import tempfile
import os

code_dir = tempfile.mkdtemp()
inference_file = os.path.join(code_dir, 'inference.py')

s3_client = boto3.client('s3', region_name=region)
s3_client.download_file(bucket, f'{s3_prefix}ml/bundle/deployment-code/inference.py', inference_file)
print(f"   Downloaded inference.py to {code_dir}")

from sagemaker.utils import name_from_base

artifact_name = model_s3_uri.rstrip('/').split('/')[-3]
model_name = name_from_base(artifact_name)
print(f"   SageMaker model name: {model_name}")

sklearn_model = SKLearnModel(
    model_data=model_data,
    role=role,
    entry_point='inference.py',
    source_dir=code_dir,
    framework_version=sklearn_version,
    py_version=python_version,
    sagemaker_session=session,
    name=model_name,
    env={'MODEL_NAME': artifact_name}
)

endpoint_name = name_from_base("realistic-classifier")
predictor = sklearn_model.deploy(
    initial_instance_count=1,
    instance_type=inference_instance_type,
    endpoint_name=endpoint_name,
)
endpoint_name = predictor.endpoint_name
print(f"   ✅ Live endpoint: {endpoint_name}")

# deploy() already created the SageMaker model. Build the batch transformer
# from that name so CreateModel is not called a second time.
from sagemaker.transformer import Transformer

batch_output_path = f"s3://{bucket}/{s3_prefix}ml/batch-output/{model_name}/"
transformer = Transformer(
    model_name=sklearn_model.name,
    instance_count=1,
    instance_type=inference_instance_type,
    output_path=batch_output_path,
    accept='text/csv',
    assemble_with='Line',
    strategy='MultiRecord',
    sagemaker_session=session,
)
print(f"   ✅ Batch transform model: {transformer.model_name}")
print(f"   Output path: {batch_output_path}")

In [ ]:
# Test live and batch inference
print("\n🧪 Testing live endpoint...")

from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import CSVDeserializer
import io
import numpy as np
import pandas as pd

predictor.serializer = CSVSerializer()
predictor.deserializer = CSVDeserializer()

# Test data with 20 features (matching training)
test_data = np.random.randn(2, 20)
live_predictions = predictor.predict(test_data.tolist())
print(f"   Test input shape: {test_data.shape[0]}x{test_data.shape[1]}")
print(f"   Live predictions: {live_predictions}")
print("   ✅ Live inference successful")

print("\n🧪 Testing batch transform...")

csv_buffer = io.StringIO()
pd.DataFrame(test_data).to_csv(csv_buffer, header=False, index=False)

if not batch_input_file or not str(batch_input_file).strip():
    raise ValueError("batch_input_file is required")
batch_input_file = str(batch_input_file).strip()

input_key = f"{s3_prefix}ml/batch-input/{model_name}/{batch_input_file}"
s3_client.put_object(Bucket=bucket, Key=input_key, Body=csv_buffer.getvalue().encode("utf-8"))
input_s3_uri = f"s3://{bucket}/{input_key}"
print(f"   Uploaded test input to {input_s3_uri}")

transform_job_name = name_from_base("realistic-classifier")
transformer.transform(
    data=input_s3_uri,
    content_type="text/csv",
    split_type="Line",
    job_name=transform_job_name,
    wait=True,
    logs=False,
)

output_key = f"{s3_prefix}ml/batch-output/{model_name}/{batch_input_file}.out"
output_obj = s3_client.get_object(Bucket=bucket, Key=output_key)
batch_predictions = output_obj["Body"].read().decode("utf-8")
print(f"   Transform job: {transformer.latest_transform_job.name}")
print(f"   Batch predictions:\n{batch_predictions}")
print("   ✅ Batch inference successful")